# Domineering en tablero de 3 × 4
**Elizabeth Chungara Choque · SIS420 · Inteligencia Artificial**

## ¿Qué es Domineering?

Es un juego de estrategia para dos jugadores que comparten un tablero de tres filas y cuatro columnas. Cada ficha ocupa dos casillas vecinas. El objetivo es dejar al rival sin espacio para colocar su siguiente ficha.

## Reglas y turnos

1. Se empieza con las doce casillas vacías. En este proyecto, Vertical juega primero.
2. Vertical coloca una ficha en dos casillas libres de la misma columna, una debajo de la otra. Horizontal coloca la suya en dos casillas libres de la misma fila, una al lado de la otra.
3. Los jugadores se alternan. Cada ficha permanece donde fue colocada. No se permite cubrir otra ficha, jugar en diagonal ni salir del tablero.
4. Quien empieza su turno sin una jugada legal pierde. Pueden quedar casillas vacías. No hay empate.

## Una partida de ejemplo

Las casillas se numeran por filas. Esta es una secuencia posible, con el agente como Vertical y la persona como Horizontal.

| Fila | Columna 1 | Columna 2 | Columna 3 | Columna 4 |
|---|---:|---:|---:|---:|
| Primera | 1 | 2 | 3 | 4 |
| Segunda | 5 | 6 | 7 | 8 |
| Tercera | 9 | 10 | 11 | 12 |

1. Vertical ocupa **6 y 10**. En la columna 2, la casilla 10 está justo debajo de la 6.
2. Horizontal escribe **1** y ocupa **1 y 2**. Solo tiene que indicar la casilla izquierda de su ficha.
3. Vertical ocupa **8 y 12**. Horizontal responde colocando su ficha en **3 y 4**.
4. Vertical ocupa **7 y 11**. Quedan libres **5 y 9**, una encima de la otra. Le toca a Horizontal y necesita dos casillas libres en la misma fila. **Gana Vertical.**

## Cómo se realizó el trabajo

Se adaptó la estructura del tres en raya estudiado en clase. La clase `Board` representa el tablero, calcula dónde cabe cada ficha y determina quién gana según el turno. La clase `Game` organiza las partidas entre dos agentes. Cada `Agent` guarda las posiciones visitadas y, al terminar la partida, actualiza su tabla según la recompensa recibida. Ambos entrenan durante 10 000 partidas. Después se guarda la tabla de Vertical para que una persona pueda jugar contra él en la última sección.


In [1]:
import pickle
import numpy as np
import pandas as pd

## 1. Board · Tablero y reglas
Una acción se identifica por su casilla inicial. Por ejemplo, desde la esquina superior izquierda Vertical ocupa dos filas y Horizontal ocupa dos columnas. El tablero almacena 0 para vacío, 1 para Vertical y −1 para Horizontal.

In [2]:
class Board:
    def __init__(self):
        self.state = np.zeros((3, 4), dtype=np.int8)

    def reset(self):
        self.state.fill(0)

    def cells(self, symbol, row, col):
        if symbol == 1:
            return (row, col), (row + 1, col)
        if symbol == -1:
            return (row, col), (row, col + 1)
        raise ValueError("La orientación debe ser 1 o -1.")

    def valid_moves(self, symbol):
        moves = []
        for row in range(3):
            for col in range(4):
                start, end = self.cells(symbol, row, col)
                if end[0] < 3 and end[1] < 4:
                    if self.state[start] == 0 and self.state[end] == 0:
                        moves.append(start)
        return moves

    def update(self, symbol, row, col):
        if (row, col) not in self.valid_moves(symbol):
            raise ValueError("La ficha no cabe o se superpone con otra.")
        for cell in self.cells(symbol, row, col):
            self.state[cell] = symbol

    def is_game_over(self, symbol):
        if not self.valid_moves(symbol):
            return -symbol
        return None

## 2. Game · Dos agentes juegan entre sí
En cada episodio ambos registran el tablero después de cada jugada. Al pasar el turno comprobamos si el siguiente jugador puede colocar su ficha. El ganador recibe 1 y el perdedor 0.

In [3]:
class Game:
    def __init__(self, vertical, horizontal):
        self.players = {1: vertical, -1: horizontal}
        for symbol, player in self.players.items():
            player.symbol = symbol
        self.board = Board()

    def selfplay(self, rounds):
        wins = {1: 0, -1: 0}
        for _ in range(rounds):
            self.board.reset()
            for player in self.players.values():
                player.reset()
            turn = 1
            winner = self.board.is_game_over(turn)
            while winner is None:
                row, col = self.players[turn].move(self.board)
                self.board.update(turn, row, col)
                for player in self.players.values():
                    player.update(self.board)
                turn = -turn
                winner = self.board.is_game_over(turn)
            for symbol, player in self.players.items():
                player.reward(int(symbol == winner))
            wins[winner] += 1
        return wins

## 3. Agent · Elegir, registrar y aprender
Al explorar, el agente elige una jugada legal al azar. En los demás turnos simula cada opción y consulta el valor del tablero resultante. La clave contiene las doce casillas en orden de filas. Cada agente mantiene su propia tabla.

Al terminar recorre su historial desde la última posición. En cada paso aplica $V(s) \leftarrow V(s)+\alpha[objetivo-V(s)]$. El primer objetivo es la recompensa final. Después, el valor recién actualizado se convierte en el objetivo de la posición anterior.

In [4]:
class Agent:
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.alpha = alpha
        self.prob_exp = prob_exp
        self.positions = []
        self.value_function = {}

    def reset(self):
        self.positions.clear()

    def move(self, board, explore=True):
        moves = board.valid_moves(self.symbol)
        if explore and np.random.random() < self.prob_exp:
            return moves[np.random.randint(len(moves))]
        best_value = -1
        for row, col in moves:
            candidate = board.state.copy()
            for cell in board.cells(self.symbol, row, col):
                candidate[cell] = self.symbol
            value = self.value_function.get(tuple(candidate.ravel()), 0)
            if value >= best_value:
                best_value = value
                best_move = (row, col)
        return best_move

    def update(self, board):
        self.positions.append(tuple(board.state.ravel()))

    def reward(self, reward):
        for state in reversed(self.positions):
            value = self.value_function.get(state, 0)
            value += self.alpha * (reward - value)
            self.value_function[state] = value
            reward = value

## 4. Entrenamiento e inspección
Usamos 10 000 partidas para facilitar la ejecución. Puedes cambiar esa cantidad en la celda. Cada ejecución empieza con agentes nuevos. Los conteos incluyen decisiones exploratorias durante el entrenamiento.

In [5]:
np.random.seed(42)
vertical = Agent()
horizontal = Agent()
game = Game(vertical, horizontal)
partidas = 10_000
wins = game.selfplay(partidas)
print("Partidas:", partidas)
print("Victorias de Vertical:", wins[1])
print("Victorias de Horizontal:", wins[-1])
assert sum(wins.values()) == partidas

Partidas: 10000
Victorias de Vertical: 2929
Victorias de Horizontal: 7071


In [6]:
tabla = pd.DataFrame(vertical.value_function.items(), columns=["Estado", "Valor"])
tabla.sort_values("Valor", ascending=False).head(10)

                                       Estado  Valor
4    (0, 1, -1, -1, 1, 1, 0, 1, 1, -1, -1, 1)    1.0
128  (0, 1, -1, -1, 0, 1, 1, 0, -1, -1, 1, 0)    1.0
129    (0, 1, 0, 0, 0, 1, 1, 0, -1, -1, 1, 0)    1.0
169    (-1, -1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1)    1.0
188    (0, 0, -1, -1, 0, 1, 1, 0, 0, 1, 1, 0)    1.0
181  (-1, -1, 1, 0, 1, 1, 1, 0, 1, 1, -1, -1)    1.0
127    (0, -1, -1, 0, 1, 0, 1, 0, 1, 0, 1, 0)    1.0
76   (-1, -1, -1, -1, 1, 0, 1, 0, 1, 0, 1, 0)    1.0
47   (-1, -1, -1, -1, 1, 1, 1, 0, 1, 1, 1, 0)    1.0
51     (0, -1, -1, 0, 0, 1, 0, 1, 0, 1, 0, 1)    1.0


## 5. Guardar y recuperar la tabla de Vertical
El archivo se crea en la carpeta de ejecución. Para la partida humana reutilizamos la decisión del agente con la exploración desactivada.

In [6]:
ruta = "domineering_vertical.pickle"
with open(ruta, "wb") as archivo:
    pickle.dump(vertical.value_function, archivo)

agente = Agent()
agente.symbol = 1
with open(ruta, "rb") as archivo:
    agente.value_function = pickle.load(archivo)

## 6. Jugar contra el agente
Las casillas libres se numeran del 1 al 12 por filas. Tú juegas con Horizontal. Introduce el número de la casilla izquierda de tu ficha entre las opciones mostradas. Por ejemplo, elegir 6 ocupa 6 y 7 si ambas están libres.

In [7]:
def dibujar(board):
    for row in range(3):
        values = []
        for col in range(4):
            mark = board.state[row, col]
            label = {1: "V", -1: "H"}.get(mark, str(row * 4 + col + 1))
            values.append(f"{label:>2}")
        print(" | ".join(values))
    print()

def jugar(agente):
    board = Board()
    turn = 1
    dibujar(board)
    while board.is_game_over(turn) is None:
        if turn == 1:
            row, col = agente.move(board, explore=False)
            print("Vertical empieza su ficha en", row * 4 + col + 1)
        else:
            options = {r * 4 + c + 1: (r, c) for r, c in board.valid_moves(turn)}
            while True:
                choice = input(f"Horizontal, elige entre {list(options)}: ").strip()
                if choice in {str(number) for number in options}:
                    row, col = options[int(choice)]
                    break
                print("Elige una de las casillas indicadas.")
        board.update(turn, row, col)
        dibujar(board)
        turn = -turn
    print("Gana Vertical." if turn == -1 else "Gana Horizontal.")

Para iniciar una partida, cambia `False` por `True` y ejecuta esta última celda. Puedes repetirla para jugar otra vez.

In [8]:
jugar_ahora = False
if jugar_ahora:
    jugar(agente)